In [2]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = "RR_scheduler_log.csv"
OUTPUT_FILE = "RR_clean.csv"

STARTUP_ROWS = 500

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)
print(f"Rows: {len(df):,}")
print(f"UEs : {df['ue_id'].nunique()}")

# ============================================================
# 1. TRIM STARTUP TRANSIENT
# ============================================================

print("\n" + "=" * 70)
print("1. STARTUP TRIMMING")
print("=" * 70)

before = len(df)

# Remove first 500 logged rows
df = df.iloc[STARTUP_ROWS:].reset_index(drop=True)

print(f"Removed startup rows : {before - len(df):,}")
print(f"Remaining rows       : {len(df):,}")

# ============================================================
# 2. REMOVE NaN / INF
# ============================================================

print("\n" + "=" * 70)
print("2. NaN / INF CLEANING")
print("=" * 70)

before = len(df)

df = df.replace([np.inf, -np.inf], np.nan)

nan_rows = df.isnull().any(axis=1).sum()

df = df.dropna().reset_index(drop=True)

print(f"Rows containing NaN/Inf removed : {nan_rows:,}")
print(f"Remaining rows                  : {len(df):,}")

# ============================================================
# 3. PHYSICAL VALIDITY CHECKS
# ============================================================

print("\n" + "=" * 70)
print("3. PHYSICAL VALIDITY CHECKS")
print("=" * 70)

before = len(df)

valid = (
    df["reported_cqi"].between(1, 15)
    & df["mcs"].between(0, 27)
    & (df["buffer"] >= 0)
    & (df["avg_rate"] >= 0)
    & (df["estimated_rate"] >= 0)
    & (df["allocated_prbs"] >= 0)
    & (df["allocated_bytes"] >= 0)
    & df["scheduled"].isin([0, 1])
)

df = df[valid].copy()

print(f"Invalid physical rows removed : {before - len(df):,}")
print(f"Remaining rows                : {len(df):,}")

# ============================================================
# 4. PF / RR PRIORITY INSPECTION
# ============================================================

print("\n" + "=" * 70)
print("4. PF / RR PRIORITY")
print("=" * 70)

# IMPORTANT:
# Do NOT remove large PF or RR priority values.
# They can represent legitimate scheduler states.

print("High PF/priority values are retained.")

print(f"\nPF metric:")
print(f"  min = {df['pf_metric'].min():.4f}")
print(f"  max = {df['pf_metric'].max():.4f}")

print(f"\nRR priority:")
print(f"  min = {df['rr_priority'].min()}")
print(f"  max = {df['rr_priority'].max()}")

# ============================================================
# 4B. NUMERICAL RANGE INSPECTION
# ============================================================

print("\n" + "=" * 70)
print("NUMERICAL RANGE INSPECTION")
print("=" * 70)

check_cols = [
    "buffer",
    "avg_rate",
    "estimated_rate",
    "pf_metric",
    "rr_priority",
    "allocated_prbs",
    "allocated_bytes",
    "reward"
]

for col in check_cols:

    values = df[col].to_numpy(dtype=np.float64)

    print(
        f"{col:<20} "
        f"min={np.min(values):.6g} "
        f"max={np.max(values):.6g} "
        f"finite={np.all(np.isfinite(values))}"
    )


# ============================================================
# 5. CREATE TRAINING-SAFE FEATURES
# ============================================================

print("\n" + "=" * 70)
print("5. TRAINING FEATURES")
print("=" * 70)

# Keep original physical quantities.
# Create transformed versions for neural-network input.

df["pf_metric_log"] = np.log1p(df["pf_metric"])

df["buffer_log"] = np.log1p(df["buffer"])

df["estimated_rate_log"] = np.log1p(
    df["estimated_rate"]
)

df["avg_rate_log"] = np.log1p(
    df["avg_rate"]
)

print("Created:")
print("  pf_metric_log")
print("  buffer_log")
print("  estimated_rate_log")
print("  avg_rate_log")

# ============================================================
# 6. PRESERVE NATURAL SCHEDULING DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("6. SCHEDULING DISTRIBUTION")
print("=" * 70)

counts = df["scheduled"].value_counts().sort_index()

print(counts)

scheduled_0 = (df["scheduled"] == 0).sum()
scheduled_1 = (df["scheduled"] == 1).sum()

print()
print(f"Not scheduled : {scheduled_0:,}")
print(f"Scheduled     : {scheduled_1:,}")

print()
print("NO BALANCING PERFORMED.")
print("Natural scheduler distribution is retained.")

# ============================================================
# 7. FINAL DATASET CHECK
# ============================================================

print("\n" + "=" * 70)
print("7. FINAL DATASET")
print("=" * 70)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")
print(f"UEs     : {df['ue_id'].nunique()}")

# Check again for NaN / Inf
print(f"\nNaN values : {df.isna().sum().sum()}")

numeric_cols = df.select_dtypes(include=[np.number]).columns

print(
    f"Inf values : "
    f"{np.isinf(df[numeric_cols].to_numpy()).sum()}"
)

# ============================================================
# 8. SAVE
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("CLEANING COMPLETE")
print("=" * 70)

print(f"Saved to : {OUTPUT_FILE}")

ORIGINAL DATASET
Rows: 641,652
UEs : 5

1. STARTUP TRIMMING
Removed startup rows : 500
Remaining rows       : 641,152

2. NaN / INF CLEANING
Rows containing NaN/Inf removed : 0
Remaining rows                  : 641,152

3. PHYSICAL VALIDITY CHECKS
Invalid physical rows removed : 0
Remaining rows                : 641,152

4. PF / RR PRIORITY
High PF/priority values are retained.

PF metric:
  min = 0.2577
  max = 99.0488

RR priority:
  min = 1
  max = 6

NUMERICAL RANGE INSPECTION
buffer               min=28 max=9.99995e+06 finite=True
avg_rate             min=33.3401 max=1387.66 finite=True
estimated_rate       min=193 max=5889 finite=True
pf_metric            min=0.257707 max=99.0488 finite=True
rr_priority          min=1 max=6 finite=True
allocated_prbs       min=0 max=51 finite=True
allocated_bytes      min=0 max=5889 finite=True
reward               min=0 max=5889 finite=True

5. TRAINING FEATURES
Created:
  pf_metric_log
  buffer_log
  estimated_rate_log
  avg_rate_log

6. SCHEDU

In [5]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = "QOS_scheduler_log.csv"
OUTPUT_FILE = "QOS_clean.csv"
STARTUP_ROWS = 500
QOS_PRIORITY_MAX = 1000.0

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)
print(f"Rows : {len(df):,}")
print(f"UEs  : {df['ue_id'].nunique()}")

# ============================================================
# 1. STARTUP + NaN/INF CLEANING
# ============================================================

df = df.iloc[STARTUP_ROWS:].reset_index(drop=True)

df = df.replace([np.inf, -np.inf], np.nan)

nan_rows = df.isnull().any(axis=1).sum()
df = df.dropna().reset_index(drop=True)

print("\n" + "=" * 70)
print("1. CLEANING")
print("=" * 70)
print(f"Startup rows removed : {STARTUP_ROWS:,}")
print(f"NaN/Inf rows removed: {nan_rows:,}")
print(f"Remaining rows      : {len(df):,}")

# ============================================================
# 2. PHYSICAL VALIDITY
# ============================================================

before = len(df)

valid = (
    df["reported_cqi"].between(1, 15)
    & df["mcs"].between(0, 27)
    & (df["buffer"] >= 0)
    & (df["avg_rate"] >= 0)
    & (df["estimated_rate"] >= 0)
    & (df["allocated_prbs"] >= 0)
    & (df["allocated_bytes"] >= 0)
    & df["scheduled"].isin([0, 1])
)

df = df[valid].copy()

print("\n" + "=" * 70)
print("2. PHYSICAL VALIDITY")
print("=" * 70)
print(f"Invalid rows removed : {before - len(df):,}")
print(f"Remaining rows       : {len(df):,}")

# ============================================================
# 3. TRAINING FEATURES
# ============================================================

df["qos_priority_train"] = df["qos_priority"].clip(
    upper=QOS_PRIORITY_MAX
)

df["qos_priority_scaled"] = pd.qcut(
    df["qos_priority_train"],
    q=6,
    labels=[1, 2, 3, 4, 5, 6],
    duplicates="drop"
).astype(float)

df["pf_metric_log"] = np.log1p(df["pf_metric"])
df["buffer_log"] = np.log1p(df["buffer"])
df["estimated_rate_log"] = np.log1p(df["estimated_rate"])
df["avg_rate_log"] = np.log1p(df["avg_rate"])

capped_count = (
    df["qos_priority"] > QOS_PRIORITY_MAX
).sum()

print("\n" + "=" * 70)
print("3. TRAINING FEATURES")
print("=" * 70)
print(f"QOS priority cap : {QOS_PRIORITY_MAX:.0f}")
print(f"Values capped    : {capped_count:,}")

print("Created:")
print("  qos_priority_train")
print("  qos_priority_scaled")
print("  pf_metric_log")
print("  buffer_log")
print("  estimated_rate_log")
print("  avg_rate_log")

# ============================================================
# 4. PRIORITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("4. PRIORITY CHECK")
print("=" * 70)

print("Raw QOS priority:")
print(f"  min    : {df['qos_priority'].min():.6e}")
print(f"  median : {df['qos_priority'].median():.6e}")
print(f"  max    : {df['qos_priority'].max():.6e}")

print("\nTraining QOS priority:")
print(f"  min    : {df['qos_priority_train'].min():.6f}")
print(f"  max    : {df['qos_priority_train'].max():.6f}")

print("\nScaled QOS priority:")
print(df["qos_priority_scaled"].value_counts().sort_index())

print("\nRR/QoS priority ranges:")
print(f"QoS scaled : {df['qos_priority_scaled'].min():.0f} - "
      f"{df['qos_priority_scaled'].max():.0f}")

# ============================================================
# 5. SCHEDULING DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("5. SCHEDULING DISTRIBUTION")
print("=" * 70)

counts = df["scheduled"].value_counts().sort_index()

print(f"Not scheduled : {counts.get(0, 0):,}")
print(f"Scheduled     : {counts.get(1, 0):,}")
print("NO BALANCING PERFORMED.")

# ============================================================
# 6. NUMERICAL SAFETY CHECK
# ============================================================

numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_array = df[numeric_cols].to_numpy()

nan_total = np.isnan(numeric_array).sum()
inf_total = np.isinf(numeric_array).sum()

print("\n" + "=" * 70)
print("6. NUMERICAL SAFETY CHECK")
print("=" * 70)
print(f"NaN values : {nan_total:,}")
print(f"Inf values : {inf_total:,}")

if nan_total == 0 and inf_total == 0:
    print("Numerical safety check passed ✓")
else:
    print("WARNING: Numerical issues remain.")

# ============================================================
# 7. FINAL DATASET
# ============================================================

print("\n" + "=" * 70)
print("7. FINAL DATASET")
print("=" * 70)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")
print(f"UEs     : {df['ue_id'].nunique()}")

print("\nQOS priority:")
print(f"Raw maximum        : {df['qos_priority'].max():.6e}")
print(f"Training maximum   : {df['qos_priority_train'].max():.2f}")
print(f"Scaled range       : "
      f"{df['qos_priority_scaled'].min():.0f} - "
      f"{df['qos_priority_scaled'].max():.0f}")

print("\nScheduling distribution:")
print(df["scheduled"].value_counts())

# ============================================================
# 8. SAVE
# ============================================================

df.to_csv(OUTPUT_FILE, index=False)

print("\n" + "=" * 70)
print("CLEANING COMPLETE")
print("=" * 70)
print(f"Saved to : {OUTPUT_FILE}")

ORIGINAL DATASET
Rows : 508,621
UEs  : 5

1. CLEANING
Startup rows removed : 500
NaN/Inf rows removed: 0
Remaining rows      : 508,121

2. PHYSICAL VALIDITY
Invalid rows removed : 0
Remaining rows       : 508,121

3. TRAINING FEATURES
QOS priority cap : 1000
Values capped    : 0
Created:
  qos_priority_train
  qos_priority_scaled
  pf_metric_log
  buffer_log
  estimated_rate_log
  avg_rate_log

4. PRIORITY CHECK
Raw QOS priority:
  min    : 2.417030e-04
  median : 4.230370e-03
  max    : 1.218460e-01

Training QOS priority:
  min    : 0.000242
  max    : 0.121846

Scaled QOS priority:
qos_priority_scaled
1.0    84690
2.0    84689
3.0    84683
4.0    84688
5.0    84688
6.0    84683
Name: count, dtype: int64

RR/QoS priority ranges:
QoS scaled : 1 - 6

5. SCHEDULING DISTRIBUTION
Not scheduled : 342,197
Scheduled     : 165,924
NO BALANCING PERFORMED.

6. NUMERICAL SAFETY CHECK
NaN values : 0
Inf values : 0
Numerical safety check passed ✓

7. FINAL DATASET
Rows    : 508,121
Columns : 26
UE

In [2]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = "DQN_scheduler_log.csv"
OUTPUT_FILE = "DQN_clean.csv"
STARTUP_ROWS = 500

df = pd.read_csv(INPUT_FILE)

# Sort chronologically before trimming startup samples
df = df.sort_values(
    ["system_slot", "frame", "slot", "ue_id"]
).reset_index(drop=True)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)
print(f"Rows : {len(df):,}")
print(f"UEs  : {df['ue_id'].nunique()}")


# ============================================================
# 1. CONVERT NUMERIC COLUMNS
# ============================================================

numeric_cols = [
    "system_slot",
    "frame",
    "slot",
    "ue_id",
    "reported_cqi",
    "buffer",
    "avg_rate",
    "estimated_rate",
    "pf_metric",
    "dqn_priority",
    "scheduled",
    "wait_slots",
    "allocated_prbs",
    "allocated_bytes",
    "mcs",
    "harq",
    "reward",
    "next_reported_cqi",
    "next_buffer",
    "next_avg_rate",
    "next_estimated_rate"
]


# Convert only columns that actually exist
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# 2. STARTUP + NaN/INF CLEANING
# ============================================================

before = len(df)

df = df.iloc[STARTUP_ROWS:].reset_index(drop=True)

df = df.replace([np.inf, -np.inf], np.nan)

nan_rows = df.isnull().any(axis=1).sum()

df = df.dropna().reset_index(drop=True)

print("\n" + "=" * 70)
print("1. CLEANING")
print("=" * 70)
print(f"Startup rows removed : {min(STARTUP_ROWS, before):,}")
print(f"NaN/Inf rows removed : {nan_rows:,}")
print(f"Remaining rows       : {len(df):,}")

# ============================================================
# 3. PHYSICAL VALIDITY
# ============================================================

before = len(df)

valid = (
    df["reported_cqi"].between(1, 15)
    & df["next_reported_cqi"].between(1, 15)
    & df["mcs"].between(0, 27)
    & (df["buffer"] >= 0)
    & (df["next_buffer"] >= 0)
    & (df["avg_rate"] >= 0)
    & (df["next_avg_rate"] >= 0)
    & (df["estimated_rate"] >= 0)
    & (df["next_estimated_rate"] >= 0)
    & (df["pf_metric"] >= 0)
    & (df["allocated_prbs"] >= 0)
    & (df["allocated_bytes"] >= 0)
    & df["scheduled"].isin([0, 1])
    & np.isfinite(df["dqn_priority"])
)

df = df[valid].copy()

print("\n" + "=" * 70)
print("2. PHYSICAL VALIDITY")
print("=" * 70)
print(f"Invalid rows removed : {before - len(df):,}")
print(f"Remaining rows       : {len(df):,}")

# ============================================================
# 4. TRAINING FEATURES
# ============================================================

df["pf_metric_log"] = np.log1p(df["pf_metric"])
df["buffer_log"] = np.log1p(df["buffer"])
df["estimated_rate_log"] = np.log1p(df["estimated_rate"])
df["avg_rate_log"] = np.log1p(df["avg_rate"])

df["dqn_priority_log"] = np.sign(df["dqn_priority"]) * np.log1p(
    np.abs(df["dqn_priority"])
)


print("\n" + "=" * 70)
print("3. TRAINING FEATURES")
print("=" * 70)
print("Created:")
print("  pf_metric_log")
print("  buffer_log")
print("  estimated_rate_log")
print("  avg_rate_log")
print("  dqn_priority_log")

# ============================================================
# 5. DQN FEATURE INSPECTION
# ============================================================

print("\n" + "=" * 70)
print("4. PRIORITY INSPECTION")
print("=" * 70)

print("\nDQN priority:")
print(f"  min    = {df['dqn_priority'].min():.6f}")
print(f"  median = {df['dqn_priority'].median():.6f}")
print(f"  max    = {df['dqn_priority'].max():.6f}")

print("\nPF metric:")
print(f"  min = {df['pf_metric'].min():.6f}")
print(f"  max = {df['pf_metric'].max():.6f}")

# ============================================================
# 6. SCHEDULING DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("5. SCHEDULING DISTRIBUTION")
print("=" * 70)

counts = df["scheduled"].value_counts().sort_index()

print(f"Not scheduled : {counts.get(0, 0):,}")
print(f"Scheduled     : {counts.get(1, 0):,}")
print("NO BALANCING PERFORMED.")

# ============================================================
# 7. NUMERICAL SAFETY CHECK
# ============================================================

numeric_cols_final = df.select_dtypes(
    include=[np.number]
).columns

numeric_array = df[numeric_cols_final].to_numpy()

nan_total = np.isnan(numeric_array).sum()
inf_total = np.isinf(numeric_array).sum()

print("\n" + "=" * 70)
print("6. NUMERICAL SAFETY CHECK")
print("=" * 70)
print(f"NaN values : {nan_total:,}")
print(f"Inf values : {inf_total:,}")

if nan_total == 0 and inf_total == 0:
    print("Numerical safety check passed ✓")
else:
    print("WARNING: Numerical issues remain.")

# ============================================================
# 8. FINAL DATASET
# ============================================================

print("\n" + "=" * 70)
print("7. FINAL DATASET")
print("=" * 70)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")
print(f"UEs     : {df['ue_id'].nunique()}")

print("\nDQN priority:")
print(f"Raw maximum : {df['dqn_priority'].max():.6e}")
print(f"Log maximum : {df['dqn_priority_log'].max():.6f}")

print("\nScheduling distribution:")
print(df["scheduled"].value_counts())

# ============================================================
# 9. SAVE
# ============================================================

df.to_csv(OUTPUT_FILE, index=False)

print("\n" + "=" * 70)
print("CLEANING COMPLETE")
print("=" * 70)
print(f"Saved to : {OUTPUT_FILE}")

ORIGINAL DATASET
Rows : 830,782
UEs  : 5

1. CLEANING
Startup rows removed : 500
NaN/Inf rows removed : 0
Remaining rows       : 830,282

2. PHYSICAL VALIDITY
Invalid rows removed : 0
Remaining rows       : 830,282

3. TRAINING FEATURES
Created:
  pf_metric_log
  buffer_log
  estimated_rate_log
  avg_rate_log
  dqn_priority_log
  dqn_scale_log

4. PRIORITY INSPECTION

DQN priority:
  min    = -0.750687
  median = -0.633727
  max    = 0.607907

PF metric:
  min = 0.451510
  max = 100.000000

5. SCHEDULING DISTRIBUTION
Not scheduled : 578,441
Scheduled     : 251,841
NO BALANCING PERFORMED.

6. NUMERICAL SAFETY CHECK
NaN values : 0
Inf values : 0
Numerical safety check passed ✓

7. FINAL DATASET
Rows    : 830,282
Columns : 26
UEs     : 5

DQN priority:
Raw maximum : 6.079070e-01
Log maximum : 0.474933

Scheduling distribution:
scheduled
0    578441
1    251841
Name: count, dtype: int64

CLEANING COMPLETE
Saved to : DQN_clean.csv
